In [ ]:

MedQA: RAG + Fine-Tuned LLM for Medical Question Answering
CSE3720 — Generative AI and LLMs | Assignment 2
Department of Computer Science and Engineering, BML Munjal University

📋 PRE-REQUISITES — READ BEFORE RUNNING
Step 1: Google Colab Setup
Open this notebook in Google Colab (https://colab.research.google.com)
Go to Runtime → Change Runtime Type → T4 GPU (free tier)
Step 2: HuggingFace Account & Token
Create free account at https://huggingface.co
Go to Settings → Access Tokens → New Token (role: Read)
In Colab left sidebar → 🔑 Secrets → Add: Name=HF_TOKEN, Value=your token
Step 3: Model Access
https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2 → Click Agree
https://huggingface.co/google/gemma-2b-it → Click Agree
https://huggingface.co/tiiuae/falcon-7b-instruct → Free, no gate
Step 4: Run cells in ORDER. Do NOT skip.

**SECTION A1:** Dataset Documentation, EDA & Balanced Sampling


Purpose: This section covers the foundational data work. It downloads the MedQuAD dataset from HuggingFace and performs Exploratory Data Analysis (EDA).

EDA: It calculates statistics for question types (e.g., symptoms, treatment) and text lengths.

Stratified Sampling: To ensure stability during training, it selects a representative sample (configured to 2,000 examples) across the different medical categories.

**SECTION A2:** Dataset Formatting & Train/Val/Test Split


Purpose: This section prepares the data for an instruction-following model.

Formatting: It wraps the medical Q&A pairs into the <|system|>, <|user|>, and <|assistant|> template.

Splitting: It divides the cleaned data into Train (85%), Validation (10%), and Test (5%) sets and saves them to disk as ./processed_dataset for use in the following sections.

In [11]:

import os
import json
import random
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset, DatasetDict
from collections import Counter

# ─────────────────────────────────────────────────────────────────────────────
# 1. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CONFIG = {
    "dataset_name": "keivalya/MedQuad-MedicalQnADataset",
    "output_dir": "./processed_dataset",
    "seed": 42,
    "train_split": 0.85,
    "val_split": 0.10,
    "test_split": 0.05,
    "max_input_length": 256,      # tokens (approximate)
    "max_output_length": 512,     # tokens (approximate)
    "min_answer_length": 20,      # characters — filter very short answers
    "sample_size": 2000,          # use first 2000 examples for experiments
}

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

# ─────────────────────────────────────────────────────────────────────────────
# 2. LOAD RAW DATASET
# ─────────────────────────────────────────────────────────────────────────────
def load_raw_dataset():
    """Load dataset from HuggingFace Hub."""
    print("Loading MedQuad dataset from HuggingFace...")
    dataset = load_dataset(CONFIG["dataset_name"], split="train")
    df = dataset.to_pandas()
    print(f"  Total rows loaded: {len(df)}")
    print(f"  Columns: {list(df.columns)}")
    return df

# ─────────────────────────────────────────────────────────────────────────────
# 3. EXPLORATORY DATA ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
def explore_dataset(df: pd.DataFrame):
    """Print dataset statistics."""
    print("\n=== Dataset Statistics ===")
    print(f"Total examples  : {len(df)}")
    print(f"Columns         : {list(df.columns)}")
    print(f"\nQuestion type distribution:")
    if "qtype" in df.columns:
        for qtype, count in df["qtype"].value_counts().items():
            print(f"  {qtype:<30} {count:>5} ({count/len(df)*100:.1f}%)")

    print(f"\nQuestion length stats (characters):")
    df["q_len"] = df["Question"].str.len()
    print(df["q_len"].describe().to_string())

    print(f"\nAnswer length stats (characters):")
    df["a_len"] = df["Answer"].str.len()
    print(df["a_len"].describe().to_string())

    # Missing values
    print(f"\nMissing values:")
    print(df[["Question", "Answer"]].isnull().sum())

# ─────────────────────────────────────────────────────────────────────────────
# 4. CLEANING AND FILTERING
# ─────────────────────────────────────────────────────────────────────────────
def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Apply cleaning rules to the dataset."""
    print("\n=== Cleaning Dataset ===")
    initial = len(df)

    # Drop rows with missing values
    df = df.dropna(subset=["Question", "Answer"])
    print(f"  After dropping NaN rows         : {len(df)} (removed {initial - len(df)})")

    # Strip whitespace
    df["Question"] = df["Question"].str.strip()
    df["Answer"]   = df["Answer"].str.strip()

    # Remove duplicate (Question, Answer) pairs
    before_dedup = len(df)
    df = df.drop_duplicates(subset=["Question", "Answer"])
    print(f"  After dedup                     : {len(df)} (removed {before_dedup - len(df)})")

    # Filter very short answers
    before_filter = len(df)
    df = df[df["Answer"].str.len() >= CONFIG["min_answer_length"]]
    print(f"  After min-answer-length filter  : {len(df)} (removed {before_filter - len(df)})")

    # Remove rows where Question appears to be empty or very short
    df = df[df["Question"].str.len() >= 10]
    print(f"  After min-question-length filter: {len(df)}")

    return df.reset_index(drop=True)

# ─────────────────────────────────────────────────────────────────────────────
# 5. FORMAT INTO INSTRUCTION-FOLLOWING TEMPLATE
# ─────────────────────────────────────────────────────────────────────────────
PROMPT_TEMPLATE = (
    "<|system|>\n"
    "You are a helpful and accurate medical assistant.\n"
    "<|user|>\n{question}\n"
    "<|assistant|>\n"
)

def format_example(row: pd.Series) -> dict:
    """Convert a dataset row to an instruction-following format."""
    prompt    = PROMPT_TEMPLATE.format(question=row["Question"].strip())
    response  = row["Answer"].strip()
    full_text = prompt + response   # used for causal LM training

    return {
        "input"    : prompt,
        "output"   : response,
        "text"     : full_text,       # merged for causal LM (GPT-style)
        "qtype"    : row.get("qtype", "unknown"),
    }

def prepare_instruction_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Apply instruction template to every row."""
    print("\n=== Formatting as Instruction Dataset ===")
    formatted = df.apply(format_example, axis=1, result_type="expand")
    print(f"  Formatted {len(formatted)} examples.")
    print("\n  Sample formatted example:")
    print("  INPUT:\n" + formatted.iloc[0]["input"][:300])
    print("  OUTPUT:\n" + formatted.iloc[0]["output"][:200])
    return formatted

# ─────────────────────────────────────────────────────────────────────────────
# 6. SPLIT INTO TRAIN / VAL / TEST
# ─────────────────────────────────────────────────────────────────────────────
def split_dataset(df: pd.DataFrame, sample_size: int = None) -> DatasetDict:
    """Shuffle, optionally sample, then split."""
    if sample_size and len(df) > sample_size:
        df = df.sample(n=sample_size, random_state=CONFIG["seed"]).reset_index(drop=True)
        print(f"\n=== Using {sample_size} examples for experiments ===")

    df = df.sample(frac=1, random_state=CONFIG["seed"]).reset_index(drop=True)
    n  = len(df)
    n_train = int(n * CONFIG["train_split"])
    n_val   = int(n * CONFIG["val_split"])

    train_df = df.iloc[:n_train]
    val_df   = df.iloc[n_train:n_train + n_val]
    test_df  = df.iloc[n_train + n_val:]

    print(f"\n=== Dataset Splits ===")
    print(f"  Train : {len(train_df)}")
    print(f"  Val   : {len(val_df)}")
    print(f"  Test  : {len(test_df)}")

    dataset_dict = DatasetDict({
        "train" : Dataset.from_pandas(train_df.reset_index(drop=True)),
        "val"   : Dataset.from_pandas(val_df.reset_index(drop=True)),
        "test"  : Dataset.from_pandas(test_df.reset_index(drop=True)),
    })
    return dataset_dict

# ─────────────────────────────────────────────────────────────────────────────
# 7. SAVE
# ─────────────────────────────────────────────────────────────────────────────
def save_dataset(dataset_dict: DatasetDict):
    os.makedirs(CONFIG["output_dir"], exist_ok=True)
    dataset_dict.save_to_disk(CONFIG["output_dir"])
    print(f"\n=== Dataset saved to {CONFIG['output_dir']} ===")

    # Also save as JSONL for easy inspection
    for split_name, split_data in dataset_dict.items():
        path = os.path.join(CONFIG["output_dir"], f"{split_name}.jsonl")
        with open(path, "w") as f:
            for example in split_data:
                f.write(json.dumps(example) + "\n")
        print(f"  Saved {split_name}.jsonl ({len(split_data)} examples)")

# ─────────────────────────────────────────────────────────────────────────────
# 8. MAIN
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df = load_raw_dataset()
    explore_dataset(df)
    df = clean_dataset(df)
    df = prepare_instruction_dataset(df)
    dataset_dict = split_dataset(df, sample_size=CONFIG["sample_size"])
    save_dataset(dataset_dict)
    print("\nPreprocessing complete.")

Loading MedQuad dataset from HuggingFace...
  Total rows loaded: 16407
  Columns: ['qtype', 'Question', 'Answer']

=== Dataset Statistics ===
Total examples  : 16407
Columns         : ['qtype', 'Question', 'Answer']

Question type distribution:
  information                     4535 (27.6%)
  symptoms                        2748 (16.7%)
  treatment                       2442 (14.9%)
  inheritance                     1446 (8.8%)
  frequency                       1120 (6.8%)
  genetic changes                 1087 (6.6%)
  causes                           727 (4.4%)
  exams and tests                  653 (4.0%)
  research                         395 (2.4%)
  outlook                          361 (2.2%)
  susceptibility                   324 (2.0%)
  considerations                   235 (1.4%)
  prevention                       210 (1.3%)
  stages                            77 (0.5%)
  complications                     46 (0.3%)
  support groups                     1 (0.0%)

Question length

Saving the dataset (0/1 shards):   0%|          | 0/1700 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]


=== Dataset saved to ./processed_dataset ===
  Saved train.jsonl (1700 examples)
  Saved val.jsonl (200 examples)
  Saved test.jsonl (100 examples)

Preprocessing complete.


**SECTION A2:** QLoRA Fine-Tuning (TinyLlama-1.1B)

Quantization (QLoRA): The model is loaded in 4-bit using bitsandbytes to fit on the T4 GPU.

LoRA Configuration: Small trainable "adapters" are added to the model's layers (specifically targeting q_proj, v_proj, etc.).

Training: The Trainer runs for 1 epoch, focusing the model's weights on medical knowledge, and saves the final adapter to ./med_model.

In [16]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers", "peft", "accelerate", "bitsandbytes", "datasets"], check=True)

import os, torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments,
    Trainer, DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. CONFIG
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 2. DATA PREP (With proper labels for the Trainer)
def prepare_data(tokenizer):
    print("Loading MedQuad...")
    raw = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")
    ds = raw.shuffle(seed=42).select(range(800)) # Small sample for stability

    def tokenize_fn(batch):
        # We MUST include the EOS token at the end of the assistant response
        texts = [
            f"<|user|>\n{q}\n<|assistant|>\n{a}{tokenizer.eos_token}"
            for q, a in zip(batch["Question"], batch["Answer"])
        ]
        tokenized = tokenizer(texts, truncation=True, max_length=512, padding="max_length")
        # For the standard Trainer, 'labels' must match 'input_ids'
        tokenized["labels"] = tokenized["input_ids"].copy()
        return tokenized

    return ds.map(tokenize_fn, batched=True, remove_columns=raw.column_names)

# 3. MODEL LOADING (QLoRA)
def load_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
    model = prepare_model_for_kbit_training(model)

    # We target more modules to help the model learn better
    config = LoraConfig(
        r=8, lora_alpha=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05, task_type="CAUSAL_LM"
    )
    return get_peft_model(model, config), tokenizer

if __name__ == "__main__":
    model, tokenizer = load_model()
    dataset = prepare_data(tokenizer)

    args = TrainingArguments(
        output_dir="./med_model",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=5e-5,    # LOWER learning rate for stability
        num_train_epochs=1,
        logging_steps=10,
        fp16=True,
        save_strategy="no",
        report_to="none",
        optim="paged_adamw_32bit" # More stable for 4-bit
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    print("\n=== TRAINING ===")
    trainer.train()

    # TEST
    model.eval()
    q = "What are the symptoms of botulism?"
    prompt = f"<|user|>\n{q}\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    print("\n=== MODEL OUTPUT ===")
    with torch.no_grad():
        # use_cache=True is fine for INFERENCE, just not training
        outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True, repetition_penalty=1.2)

    # Decode only the NEW tokens
    decoded = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    print(decoded)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading MedQuad...


Map:   0%|          | 0/800 [00:00<?, ? examples/s]


=== TRAINING ===


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.698346
20,1.553195
30,1.519617
40,1.472060
50,1.471581
60,1.381292
70,1.393600
80,1.413826
90,1.339931
100,1.290078



=== MODEL OUTPUT ===
Symptoms depend on how long the toxin is left in your bloodstream. In most cases, it takes several days for Botulism to develop after eating contaminated food or drinking water containing this type of spores called Clostridium Bíthrix. The first symptom may be a nausea and vomiting that clears up within minutes. Other signs include drooping eyelids (called ptosis), difficulty swallowing words, difficulty breathing, slurring speech, muscle weakness, drooling, twitches, tetany, and respiratory failure. Most people with Botulism die from their condition. If you have already received treatment for


In [19]:
 import nltk
nltk.download('punkt_tab')
nltk.download('punkt') # Good to have both for tokenization

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [22]:
!find / -name "adapter_config.json" 2>/dev/null

/root/.cache/huggingface/hub/models--TinyLlama--TinyLlama-1.1B-Chat-v1.0/.no_exist/fe8a4ea1ffedaf415f4da2f062534de366a451e6/adapter_config.json
/content/final_medical_model/adapter_config.json


**SECTION A3:** Baseline Evaluation BEFORE Fine-Tuning


Purpose: Establishing a benchmark. Before the model learns from your specific data, this section evaluates the "out-of-the-box" TinyLlama-1.1B model.

Metric Baseline: It calculates the initial ROUGE and BLEU scores on the test set to show how a general-purpose model handles medical queries.


**SECTION A4:** Fine-Tuned Model Evaluation AFTER Fine-Tuning


Purpose: Measuring the success of the training.

Adapter Loading: This section loads the trained adapters from the previous step and attaches them back to the base model.

Post-Training Eval: It re-runs the ROUGE and BLEU metrics on the same test set used in Section B3.

**SECTION A5:** Before vs After Metrics Comparison


Purpose: This generates the Comparison Report.

Delta Calculation: It creates a table showing the percentage improvement (or "delta") between the Base model and the Fine-Tuned model for metrics like ROUGE-L and BLEU.

Latency Check: It also compares how long the model takes to generate an answer (latency) before and after tuning.

In [26]:
"""
Assignment 2: Fine-Tuning a Large Language Model Using Custom Dataset
Script 3: Evaluation — Base Model vs Fine-Tuned Model
======================================================

Computes:
  - ROUGE-1, ROUGE-2, ROUGE-L
  - BLEU score
  - BERTScore (optional)
  - Average inference latency
  - Qualitative example comparison
"""

import os
import json
import time
import torch
import numpy as np
import pandas as pd
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk

nltk.download("punkt", quiet=True)

# ─────────────────────────────────────────────────────────────────────────────
# 1. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
EVAL_CONFIG = {
    "base_model"        : "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "adapter_path"      : "/content/final_medical_model",
    "dataset_dir"       : "./processed_dataset",
    "results_dir"       : "./evaluation_results",
    "max_new_tokens"    : 256,
    "temperature"       : 0.1,
    "do_sample"         : True,
    "num_eval_samples"  : 100,   # evaluate on 100 test examples
    "seed"              : 42,
}

os.makedirs(EVAL_CONFIG["results_dir"], exist_ok=True)

PROMPT_TEMPLATE = (
    "<|system|>\n"
    "You are a helpful and accurate medical assistant.\n"
    "<|user|>\n{question}\n"
    "<|assistant|>\n"
)

# ─────────────────────────────────────────────────────────────────────────────
# 2. LOAD MODELS
# ─────────────────────────────────────────────────────────────────────────────
def load_4bit_base(model_name: str):
    """Load the base model in 4-bit (no adapters)."""
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name, quantization_config=bnb, device_map="auto", trust_remote_code=True
    )
    model.eval()
    return model, tokenizer

def load_finetuned(base_model_name: str, adapter_path: str):
    """Load fine-tuned model = base + LoRA adapter."""
    model, tokenizer = load_4bit_base(base_model_name)
    model = PeftModel.from_pretrained(model, adapter_path)
    model.eval()
    return model, tokenizer

# ─────────────────────────────────────────────────────────────────────────────
# 3. INFERENCE
# ─────────────────────────────────────────────────────────────────────────────
def generate_answer(model, tokenizer, question: str, config: dict) -> tuple[str, float]:
    """Generate an answer for a question; returns (answer_text, latency_sec)."""
    prompt = PROMPT_TEMPLATE.format(question=question.strip())
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    t0 = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens  = config["max_new_tokens"],
            temperature     = config["temperature"],
            do_sample       = config["do_sample"],
            pad_token_id    = tokenizer.eos_token_id,
        )
    latency = time.perf_counter() - t0

    # Decode only the newly generated tokens
    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    answer    = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return answer, latency

# ─────────────────────────────────────────────────────────────────────────────
# 4. METRICS
# ─────────────────────────────────────────────────────────────────────────────
def compute_rouge(predictions: list[str], references: list[str]) -> dict:
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    scores = {"rouge1": [], "rouge2": [], "rougeL": []}
    for pred, ref in zip(predictions, references):
        s = scorer.score(ref, pred)
        scores["rouge1"].append(s["rouge1"].fmeasure)
        scores["rouge2"].append(s["rouge2"].fmeasure)
        scores["rougeL"].append(s["rougeL"].fmeasure)
    return {k: float(np.mean(v)) for k, v in scores.items()}

def compute_bleu(predictions: list[str], references: list[str]) -> float:
    smoother = SmoothingFunction().method1
    bleu_scores = []
    for pred, ref in zip(predictions, references):
        ref_tokens  = nltk.word_tokenize(ref.lower())
        pred_tokens = nltk.word_tokenize(pred.lower())
        score = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoother)
        bleu_scores.append(score)
    return float(np.mean(bleu_scores))

# ─────────────────────────────────────────────────────────────────────────────
# 5. FULL EVALUATION LOOP
# ─────────────────────────────────────────────────────────────────────────────
def evaluate_model(model, tokenizer, test_examples: list[dict], config: dict,
                   label: str) -> dict:
    """Run inference + compute metrics for a model on test_examples."""
    print(f"\n=== Evaluating: {label} ===")
    predictions, latencies = [], []

    for i, ex in enumerate(test_examples):
        pred, lat = generate_answer(model, tokenizer, ex["input"], config)
        predictions.append(pred)
        latencies.append(lat)
        if (i + 1) % 10 == 0:
            print(f"  [{i+1}/{len(test_examples)}] avg latency so far: {np.mean(latencies):.2f}s")

    references = [ex["output"] for ex in test_examples]
    rouge      = compute_rouge(predictions, references)
    bleu       = compute_bleu(predictions, references)

    results = {
        "model"             : label,
        "rouge1"            : rouge["rouge1"],
        "rouge2"            : rouge["rouge2"],
        "rougeL"            : rouge["rougeL"],
        "bleu"              : bleu,
        "avg_latency_sec"   : float(np.mean(latencies)),
        "predictions"       : predictions[:10],   # store first 10 for qualitative review
        "references"        : references[:10],
    }
    print(f"  ROUGE-1: {rouge['rouge1']:.4f} | ROUGE-2: {rouge['rouge2']:.4f} | "
          f"ROUGE-L: {rouge['rougeL']:.4f} | BLEU: {bleu:.4f}")
    return results

# ─────────────────────────────────────────────────────────────────────────────
# 6. COMPARISON REPORT
# ─────────────────────────────────────────────────────────────────────────────
def print_comparison(base_results: dict, ft_results: dict):
    print("\n" + "="*60)
    print("           BASE MODEL vs FINE-TUNED MODEL")
    print("="*60)
    metrics = ["rouge1", "rouge2", "rougeL", "bleu", "avg_latency_sec"]
    for m in metrics:
        bv = base_results[m]
        fv = ft_results[m]
        if m == "avg_latency_sec":
            print(f"  {m:<20}: Base={bv:.3f}s  FT={fv:.3f}s")
        else:
            delta = (fv - bv) / max(bv, 1e-9) * 100
            arrow = "▲" if fv > bv else "▼"
            print(f"  {m:<20}: Base={bv:.4f}  FT={fv:.4f}  {arrow}{abs(delta):.1f}%")
    print("="*60)

    print("\n--- Qualitative Examples ---")
    for i in range(min(3, len(base_results["predictions"]))):
        print(f"\nExample {i+1}:")
        print(f"  Q  : {ft_results.get('questions', ['']*(i+1))[i][:100]}...")
        print(f"  Ref: {base_results['references'][i][:150]}...")
        print(f"  Base Answer : {base_results['predictions'][i][:150]}...")
        print(f"  FT Answer   : {ft_results['predictions'][i][:150]}...")

def save_results(base_results: dict, ft_results: dict, config: dict):
    output = {
        "config"        : config,
        "base_model"    : base_results,
        "finetuned"     : ft_results,
    }
    # Remove large lists for the summary file
    for key in ["predictions", "references"]:
        output["base_model"].pop(key, None)
        output["finetuned"].pop(key, None)

    path = os.path.join(config["results_dir"], "evaluation_summary.json")
    with open(path, "w") as f:
        json.dump(output, f, indent=2)
    print(f"\nResults saved to {path}")

# ─────────────────────────────────────────────────────────────────────────────
# 7. MAIN
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    torch.manual_seed(EVAL_CONFIG["seed"])

    # Load test set
    dataset = load_from_disk(EVAL_CONFIG["dataset_dir"])
    test_examples = dataset["test"].to_list()[:EVAL_CONFIG["num_eval_samples"]]
    print(f"Evaluating on {len(test_examples)} test examples.")

    # Base model
    print("\nLoading BASE model...")
    base_model, base_tok = load_4bit_base(EVAL_CONFIG["base_model"])
    base_results = evaluate_model(base_model, base_tok, test_examples, EVAL_CONFIG, label="Base")
    del base_model   # free GPU memory

    # Fine-tuned model
    print("\nLoading FINE-TUNED model...")
    ft_model, ft_tok = load_finetuned(EVAL_CONFIG["base_model"], EVAL_CONFIG["adapter_path"])
    ft_results = evaluate_model(ft_model, ft_tok, test_examples, EVAL_CONFIG, label="Fine-Tuned")

    # Compare and save
    print_comparison(base_results, ft_results)
    save_results(base_results, ft_results, EVAL_CONFIG)
    print("\nEvaluation complete.")

Evaluating on 100 test examples.

Loading BASE model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


=== Evaluating: Base ===
  [10/100] avg latency so far: 9.61s
  [20/100] avg latency so far: 8.83s
  [30/100] avg latency so far: 8.25s
  [40/100] avg latency so far: 8.18s
  [50/100] avg latency so far: 8.67s
  [60/100] avg latency so far: 8.46s
  [70/100] avg latency so far: 8.39s
  [80/100] avg latency so far: 8.33s
  [90/100] avg latency so far: 8.13s
  [100/100] avg latency so far: 8.38s
  ROUGE-1: 0.2421 | ROUGE-2: 0.0638 | ROUGE-L: 0.1573 | BLEU: 0.0219

Loading FINE-TUNED model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


=== Evaluating: Fine-Tuned ===
  [10/100] avg latency so far: 19.04s
  [20/100] avg latency so far: 19.07s
  [30/100] avg latency so far: 19.08s
  [40/100] avg latency so far: 19.09s
  [50/100] avg latency so far: 19.12s
  [60/100] avg latency so far: 19.14s
  [70/100] avg latency so far: 19.13s
  [80/100] avg latency so far: 19.16s
  [90/100] avg latency so far: 19.14s
  [100/100] avg latency so far: 19.14s
  ROUGE-1: 0.2630 | ROUGE-2: 0.0946 | ROUGE-L: 0.1787 | BLEU: 0.0459

           BASE MODEL vs FINE-TUNED MODEL
  rouge1              : Base=0.2421  FT=0.2630  ▲8.6%
  rouge2              : Base=0.0638  FT=0.0946  ▲48.4%
  rougeL              : Base=0.1573  FT=0.1787  ▲13.6%
  bleu                : Base=0.0219  FT=0.0459  ▲109.1%
  avg_latency_sec     : Base=8.383s  FT=19.137s

--- Qualitative Examples ---

Example 1:
  Q  : ...
  Ref: What are the signs and symptoms of Bardet-Biedl syndrome 5? The Human Phenotype Ontology provides the following list of signs and symptoms for Bard

**SECTION A7:** Qualitative Output Showcase


Purpose: This is your Interactive Chatbot.

Inference Demo: It provides a set of sample medical questions (like "How is Type 2 diabetes diagnosed?") to show the model's real-world performance.

Interactive Loop: It launches a CLI loop where you can type your own medical questions to see how the model has improved its accuracy and tone.

In [28]:
"""
Assignment 2: Fine-Tuning a Large Language Model Using Custom Dataset
Script 4: Inference Demo — Interactive Medical QA Chatbot
=========================================================

Loads the fine-tuned LoRA model and runs an interactive CLI chat loop.
"""

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
INFER_CONFIG = {
    "base_model"    : "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "adapter_path"  : "/content/final_medical_model",
    "max_new_tokens": 300,
    "temperature"   : 0.3,
    "do_sample"     : True,
    "top_p"         : 0.9,
}

PROMPT_TEMPLATE = (
    "<|system|>\n"
    "You are a helpful and accurate medical assistant.\n"
    "<|user|>\n{question}\n"
    "<|assistant|>\n"
)

SAMPLE_QUESTIONS = [
    "What are the symptoms of Lyme disease?",
    "How is Type 2 diabetes diagnosed?",
    "What are the treatment options for hypertension?",
    "Who is at risk for Marburg hemorrhagic fever?",
    "How can botulism be prevented?",
]

# ─────────────────────────────────────────────────────────────────────────────
# LOAD MODEL
# ─────────────────────────────────────────────────────────────────────────────
def load_model(config: dict):
    print("Loading fine-tuned medical QA model...")
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(config["base_model"], trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        config["base_model"], quantization_config=bnb,
        device_map="auto", trust_remote_code=True
    )
    model = PeftModel.from_pretrained(model, config["adapter_path"])
    model.eval()
    print("Model loaded successfully!\n")
    return model, tokenizer

# ─────────────────────────────────────────────────────────────────────────────
# INFERENCE
# ─────────────────────────────────────────────────────────────────────────────
def answer(model, tokenizer, question: str, config: dict) -> str:
    prompt = PROMPT_TEMPLATE.format(question=question.strip())
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = config["max_new_tokens"],
            temperature    = config["temperature"],
            do_sample      = config["do_sample"],
            top_p          = config["top_p"],
            pad_token_id   = tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# ─────────────────────────────────────────────────────────────────────────────
# DEMO: SAMPLE OUTPUTS
# ─────────────────────────────────────────────────────────────────────────────
def run_demo(model, tokenizer, config: dict):
    print("="*60)
    print("  MedQA Fine-Tuned Model — Sample Outputs")
    print("="*60)
    for q in SAMPLE_QUESTIONS:
        print(f"\nQ: {q}")
        resp = answer(model, tokenizer, q, config)
        print(f"A: {resp}\n" + "-"*60)

# ─────────────────────────────────────────────────────────────────────────────
# INTERACTIVE LOOP
# ─────────────────────────────────────────────────────────────────────────────
def interactive_loop(model, tokenizer, config: dict):
    print("\n" + "="*60)
    print("  Interactive Medical QA Chatbot (type 'quit' to exit)")
    print("="*60)
    while True:
        q = input("\nYour question: ").strip()
        if q.lower() in ("quit", "exit", "q"):
            print("Goodbye!")
            break
        if not q:
            continue
        resp = answer(model, tokenizer, q, config)
        print(f"\nAnswer:\n{resp}")

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    model, tokenizer = load_model(INFER_CONFIG)
    run_demo(model, tokenizer, INFER_CONFIG)
    interactive_loop(model, tokenizer, INFER_CONFIG)

Loading fine-tuned medical QA model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully!

  MedQA Fine-Tuned Model — Sample Outputs

Q: What are the symptoms of Lyme disease?
A: What are the signs and symptoms of Lyme disease? The following are the signs and symptoms of Lyme disease. These are the most common ways that Lyme disease is diagnosed. They are also the most common ways that Lyme disease is treated. - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
------------------------------------------------------------

Q: How is Type 2 diabetes diagnosed?
A: Diabetes is diagnosed by a combination of symptoms, l

KeyboardInterrupt: Interrupted by user

In [29]:
from google.colab import drive
import shutil

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Create a permanent folder on your Drive
import os
os.makedirs('/content/drive/MyDrive/MedQA_Project', exist_ok=True)

# 3. Copy your model and results to Drive
shutil.copytree('/content/final_medical_model', '/content/drive/MyDrive/MedQA_Project/final_medical_model', dirs_exist_ok=True)
shutil.copytree('/content/evaluation_results', '/content/drive/MyDrive/MedQA_Project/evaluation_results', dirs_exist_ok=True)

print("Files backed up to Google Drive successfully!")

Mounted at /content/drive
Files backed up to Google Drive successfully!
